# Experimental streaming output parsing

`StreamParser` is experimental. Its current text-stream behavior is tested,
but its public API and error contract may change as token-ID input and serving
adapters are integrated.

It is the serving reverse path for text: it parses one contiguous
segment of model-generated text into typed output messages. Its scope is
deliberately narrow:

- **Output only.** It accepts the strict canonical grammar that
  `render_str()` emits for output messages, plus an optional trailing
  `<|wait|>`. System prompts, input messages, and full transcripts are out
  of scope.
- **Provisional versus committed.** While a message is open, its payload
  streams out as provisional `PayloadDelta` records. A message becomes
  committed only when its frame closes and validates; malformed output
  never reaches the committed queue.
- **Consumer-owned recovery.** The parser stops permanently at the first
  failure and reports exact positions and raw fragments. It never guesses
  a resynchronization point; recovery always means a new parser driven by
  the consumer.

Internally, text flows through a small pipeline:

```text
TextLexer  -->  Lexeme stream  -->  OutputMachine  -->  StreamParser state
```

The lexer recognizes control glyphs across arbitrary chunk boundaries, the
machine enforces the strict grammar, and the facade builds one typed
Pydantic model per completed message.

In [ ]:
import sys
from pathlib import Path

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").exists()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Open this notebook from inside the apertus-common repository")
src = str(repo_root / "src")
if src not in sys.path:
    sys.path.insert(0, src)
print(f"Using repository: {repo_root}")

In [ ]:
from pydantic import ValidationError

from apertus_common import (
    Assistant,
    Conversation,
    FramingError,
    GenericOutput,
    IncompleteMessageError,
    MessageValidationError,
    OutputMessage,
    ParseFailure,
    ParserStateError,
    PayloadDelta,
    StreamParseError,
    StreamParser,
    Think,
    ToolCall,
    VerifiableAnswer,
    Wait,
    render_str,
)

demo = Conversation(
    items=[
        Think(payload="the user asked for the total"),
        ToolCall(id="calc:0", name="calc", args={"expression": "19 + 23"}),
        Assistant(payload="The total is 42."),
        VerifiableAnswer(payload="42"),
        Wait(),
    ]
)
rendered = render_str(demo)
print(rendered)

## Streaming mode

Engines emit text in arbitrary chunks. A chunk may split a control glyph
and may cross payload, message, and wait boundaries; the parser's result
never depends on where the chunks fall. Between pushes, `take_deltas()`
drains provisional payload fragments and `take_completed()` drains newly
validated messages, while `messages` stays cumulative.

In [ ]:
pieces = ["<|ou", rendered[4:40]]
pieces += [rendered[index : index + 9] for index in range(40, len(rendered), 9)]
assert "".join(pieces) == rendered

streaming = StreamParser()
provisional: list[PayloadDelta] = []
committed: list[OutputMessage] = []
for piece in pieces:
    streaming.push(piece)
    provisional.extend(streaming.take_deltas())
    committed.extend(streaming.take_completed())
streaming.finish()

assert streaming.waiting
assert tuple(committed) == streaming.messages

streamed_payloads: dict[int, str] = {}
for delta in provisional:
    joined = streamed_payloads.get(delta.ordinal, "") + delta.text
    streamed_payloads[delta.ordinal] = joined
assert streamed_payloads[0] == "the user asked for the total"

for message in streaming.messages:
    print(type(message).__name__)

In [ ]:
probe = StreamParser()
probe.push("<|out|> type=memo topic=demo <|hdr|> partial payl")

assert probe.current_type == "memo"
assert dict(probe.current_headers) == {"topic": "demo"}
assert probe.current_payload == "partial pay"

probe.push("oad <|/out|>")
probe.finish()
assert probe.current_type is None
memo = probe.messages[0]
assert isinstance(memo, GenericOutput)
assert memo.payload == "partial payload"
print("unknown output types build GenericOutput:", memo.type)

## Batch mode

`StreamParser.parse(text)` is one push followed by `finish()` on a fresh
parser. Batch parsing therefore runs the exact streaming code path and
produces identical typed messages.

In [ ]:
batch = StreamParser.parse(rendered)
assert batch.messages == streaming.messages
assert batch.waiting == streaming.waiting
print(f"batch and streaming agree on {len(batch.messages)} messages")

## Failure taxonomy

Every parsing failure is a `StreamParseError` carrying a stable
`ParseFailure` reason, the source offset of the rejected unit, the message
ordinal, and the parser state:

- `FramingError`: the text violates the canonical grammar or the header
  rules.
- `IncompleteMessageError`: `finish()` was called inside a control token,
  header, payload, or separator.
- `MessageValidationError`: a well-framed message failed typed
  construction, either strict tool JSON or Pydantic validation; the
  underlying cause is chained.

`ParserStateError` is different: it flags caller misuse of a terminal
parser, never malformed model output.

In [ ]:
def failure_of(text: str) -> StreamParseError:
    probe = StreamParser()
    try:
        probe.push(text)
        probe.finish()
    except StreamParseError as error:
        assert error.parser is probe
        return error
    raise AssertionError("expected a parse failure")


framing = failure_of("<|out|> type=assistant <|hdr|> hi<|/out|>")
assert isinstance(framing, FramingError)
assert framing.reason is ParseFailure.MISMATCHED_CLOSE

incomplete = failure_of("<|out|> type=think <|hdr|> half a tho")
assert isinstance(incomplete, IncompleteMessageError)
assert incomplete.reason is ParseFailure.INCOMPLETE_PAYLOAD

tool_json = failure_of(
    '<|out|> type=tool_call id=calc:0 <|hdr|> {"name":"calc","args":NaN} <|/out|>'
)
assert isinstance(tool_json, MessageValidationError)
assert tool_json.reason is ParseFailure.INVALID_TOOL_PAYLOAD
assert isinstance(tool_json.__cause__, ValueError)

model_failure = failure_of(
    '<|out|> type=tool_call id=calc:0 <|hdr|> {"name":"not a name","args":{}} <|/out|>'
)
assert model_failure.reason is ParseFailure.MESSAGE_VALIDATION
assert isinstance(model_failure.__cause__, ValidationError)

for failure in (framing, incomplete, tool_json, model_failure):
    print(
        f"{failure.reason.value:22} offset={failure.offset:3}"
        f" ordinal={failure.message_ordinal} state={failure.state}"
    )

In [ ]:
finished = StreamParser.parse("<|wait|>")
assert finished.waiting
try:
    finished.push("more")
except ParserStateError as misuse:
    print("caller misuse:", misuse)

## Raw failure data and consumer recovery

The exception exposes the exact terminal parser as `error.parser`, so
batch callers reach the same retained state streaming callers already
hold: committed messages, undrained deltas, the uncommitted raw fragment,
and the unconsumed suffix. What happens next is entirely the consumer's
decision; the parser never resynchronizes or repairs output on its own.

In [ ]:
good_message = "<|out|> type=think <|hdr|> before the fault <|/out|>"
broken_segment = good_message + "\n\nstray text <|wait|>"

owner = StreamParser()
caught: StreamParseError | None = None
try:
    owner.push(broken_segment)
except StreamParseError as error:
    caught = error

assert caught is not None
assert caught.parser is owner
assert caught.reason is ParseFailure.UNEXPECTED_TEXT

print("committed before failure:", [type(m).__name__ for m in owner.messages])
print("provisional deltas kept :", len(owner.take_deltas()))
print("raw fragment            :", repr(caught.raw_text))
print("unconsumed suffix       :", repr(caught.unconsumed))

# The consumer chooses the recovery input. Here it decides to drop the
# stray text and parse the remainder with a new parser.
chosen_suffix = "<|wait|>"
assert caught.unconsumed.endswith(chosen_suffix)
replacement = StreamParser()
replacement.push(chosen_suffix)
replacement.finish()
assert replacement.waiting
print("recovery parser accepted the chosen suffix")

## Inner workings

Three mechanisms make the observable behavior chunk-independent:

1. **Control-prefix holdback.** The lexer keeps only the longest buffer
   suffix that could still become a control glyph, so a glyph split across
   pushes is recognized whole while ordinary text flows through
   immediately.
2. **One-character payload holdback.** The machine keeps one candidate
   payload character until the next lexeme decides whether it is payload
   or the structural space before `<|/out|>`. Boundary whitespace survives
   without any normalization.
3. **Construction at close.** A typed model is built and validated exactly
   once, when its frame closes. Until then everything is provisional, and
   after any failure the parser is permanently terminal.

The machine itself consumes source-neutral lexemes, so a future token-id
frontend reuses it unchanged.

In [ ]:
# control-prefix holdback: nothing surfaces while a glyph is incomplete
held = StreamParser()
held.push("<|ou")
assert held.current_type is None
held.push("t|> type=think <|hdr|> ok <|/out|>")
held.finish()
assert held.messages == (Think(payload="ok"),)

# payload holdback: boundary whitespace is preserved exactly
spaced = StreamParser.parse("<|out|> type=assistant <|hdr|>  wide  <|/out|>")
wide = spaced.messages[0]
assert isinstance(wide, Assistant)
assert wide.payload == " wide "

# deltas and current_payload trail the pushed text by one held character
trailing = StreamParser()
trailing.push("<|out|> type=assistant <|hdr|> abc")
assert "".join(delta.text for delta in trailing.take_deltas()) == "ab"
assert trailing.current_payload == "ab"


# split invariance: whole input, single characters, and uneven chunks
# produce identical typed messages
def parse_in_chunks(text: str, size: int) -> tuple[OutputMessage, ...]:
    chunked = StreamParser()
    for start in range(0, len(text), size):
        chunked.push(text[start : start + size])
    chunked.finish()
    return chunked.messages


whole = StreamParser.parse(rendered).messages
assert parse_in_chunks(rendered, 1) == whole
assert parse_in_chunks(rendered, 7) == whole
print("split invariance holds for", len(whole), "messages")